# Linked-Read Alignment Details

In [ ]:
indir <- "reports/data/bxstats/"
platform <- "haplotagging"

In [ ]:
IRdisplay::display_html(
    sprintf(
        "<p>%s<br>%s</p>",
        format(Sys.time(), '🗓️ %d %B, %Y 🕔 %H:%M'),
        paste("Linked-read type:", platform)
    )
)

In [ ]:
library(dplyr)
library(tidyr)
library(highcharter)
library(DT)

In [ ]:
create_colored_box <- function(value, label, color = "#aeaeaeff", width = "200px", height = "90px", digits = 2, percent = FALSE) {
  .val <- formatC(value, format = "f", big.mark = ",", digits = digits, drop0trailing = T)
  if(percent){
    .val <- paste0(.val, "%")
  }
  sprintf(
    '<div style="background-color: %s; width: %s; height: %s; display: flex; flex-direction: column; align-items: center; justify-content: center; color: white; border-radius: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.15); padding: 20px; box-sizing: border-box;"><div style="font-size: 14px; font-weight: normal; margin-bottom: 2px; margin-top: 13px; opacity: 0.9; text-transform: uppercase; letter-spacing: 1px;">%s</div><div style="font-size: 43px; font-weight: bold;">%s</div></div>',
    color, width, height, label, .val
    )
}

# Function to display boxes that wrap dynamically
show_boxes <- function(boxes, gap = "15px") {
  box_html <- paste(boxes, collapse = "\n")
  html_content <- sprintf('<div style="display: flex; flex-wrap: wrap; gap: %s; width: 100%%;">%s</div>', gap, box_html)
  IRdisplay::display_html(html_content)
}

In [ ]:
iframe_height <- function(x){
  if (is.data.frame(x)) {
    frameheight <- 200 + (50 * min(10, nrow(x)))
  } else {
    frameheight <- x
  }

  sprintf("
      function(el, x) {
        // Try to resize parent iframe
        if (window.frameElement) {
          window.frameElement.style.width = '100%%';
          window.frameElement.style.height = '%spx';
        }
      }
    ", frameheight)
}

trunc_digits <- function(x, digits = 0){
  x-x%%10^-digits
}

In [ ]:
NX <- function(lengths, X=50){
  lengths <- as.numeric(sort(lengths, decreasing=TRUE))
  index <- which(cumsum(lengths) / sum(lengths) >= X/100)[1]
  return(lengths[index])
}

avg_percent_of <- function(target, other){
  # omit zeroes, take the means of target and other, then get a percent
  x <- target[target != 0]
  y <- other[other != 0]
  return(
    round(mean(x/(x+y)) * 100, 2)
  )
}

std_error <- function(x){
  sd(x)/sqrt(length((x)))
}

process_input <- function(infile){
  samplename <- gsub(".bxstats.gz", "", basename(infile))
  tb <- read.table(infile, header = T, sep = "\t") %>% select(-4, -5)
  tb$valid <- tb$molecule
  tb[tb$valid != "invalidBX", "valid"] <- "validBX"
  tb$valid <- gsub("BX", " BX", tb$valid)
  # isolate non-singletons b/c molecules with 1 read pair aren't linked reads
  multiplex_df <- filter(tb, valid == "valid BX", reads >= 2)
  singletons <- sum(tb$reads < 2 & tb$valid == "valid BX")
  tot_uniq_bx <- read.table(infile, header = F, sep = "\n", as.is = T, skip = nrow(tb) + 1, comment.char = "+")
  tot_uniq_bx <- gsub("#total unique barcodes: ", "", tot_uniq_bx$V1[1]) |> as.integer()
  tot_mol <- sum(tb$valid == "valid BX")
  tot_valid_reads <- sum(tb[tb$valid == "valid BX", "reads"])
  avg_reads_per_mol <- round(mean(multiplex_df$reads),1)
  sem_reads_per_mol <- round(std_error(multiplex_df$reads), 2)
  tot_invalid_reads <- sum(tb[tb$valid == "invalid BX", "reads"])
  avg_mol_cov <- round(mean(multiplex_df$coverage_inserts), 2)
  sem_mol_cov <- round(std_error(multiplex_df$coverage_inserts), 4)
  avg_mol_cov_bp <- round(mean(multiplex_df$coverage_bp), 2)
  sem_mol_cov_bp<- round(std_error(multiplex_df$coverage_bp), 4)
  n50 <- NX(multiplex_df$length_inferred, 50)
  n75 <- NX(multiplex_df$length_inferred, 75)
  n90 <- NX(multiplex_df$length_inferred, 90)
  datarow <- data.frame(
    sample = samplename,
    totalreads = tot_valid_reads + tot_invalid_reads,
    totaluniquemol = tot_mol,
    singletons = singletons,
    nonsingletons = nrow(multiplex_df),
    totaluniquebx = tot_uniq_bx,
    molecule2bxratio = round(tot_mol / tot_uniq_bx,2),
    totalvalidreads = tot_valid_reads,
    totalinvalidreads = tot_invalid_reads,
    totalvalidpercent = round(tot_valid_reads/(tot_valid_reads + tot_invalid_reads) * 100,2),
    averagereadspermol = avg_reads_per_mol,
    sereadspermol = sem_reads_per_mol,
    averagemolcov = avg_mol_cov,
    semolcov = sem_mol_cov,
    averagemolcovbp = avg_mol_cov_bp,
    semolcovbp = sem_mol_cov_bp,
    N50 = n50,
    N75 = n75,
    N90 = n90
  )
  return(datarow)
}

In [ ]:
infiles <- list.files(indir, "bxstats.gz", full.names = TRUE)
aggregate_df <- Reduce(rbind, Map(process_input, infiles))
aggregate_df[is.na(aggregate_df)] <- 0
n_samples <- nrow(aggregate_df)

if(nrow(aggregate_df) == 0){
  IRdisplay::display_html("<p>All input files were empty, no report to show</p>")
  q(save = "no", status = 0)
}

This report aggregates the barcode-specific information from the alignments
that were created using `harpy align`. Detailed information for any one sample
can be found in that sample's individual report.

In [ ]:
show_boxes(c(
  create_colored_box(nrow(aggregate_df), "Samples"),
  create_colored_box(avg_percent_of(aggregate_df$nonsingletons, aggregate_df$singletons), "Avg Linked", ifelse(avg_percent_of(aggregate_df$nonsingletons, aggregate_df$singletons) < 40, "#f6ab3c", "#68ae6bff"), digits = 0, percent = TRUE),
  create_colored_box(mean(aggregate_df$totalvalidpercent[aggregate_df$totalvalidpercent != 0]), "Avg Valid BX", ifelse(mean(aggregate_df$totalvalidpercent[aggregate_df$totalvalidpercent != 0]) < 40, "#f6ab3c", "#68ae6bff"), digits = 0, percent = TRUE),
  create_colored_box(mean(aggregate_df$averagereadspermol[aggregate_df$averagereadspermol != 0]), "Avg Reads/Mol", ifelse(mean(aggregate_df$averagereadspermol[aggregate_df$averagereadspermol != 0]) < 2, "#f6ab3c", "#68ae6bff")),
  create_colored_box(mean(aggregate_df$N50[aggregate_df$N50 != 0]), "N50", width = "250px", digits = 0),
  create_colored_box(mean(aggregate_df$N75[aggregate_df$N75 != 0]), "N75", width = "250px", digits = 0),
  create_colored_box(mean(aggregate_df$N90[aggregate_df$N90 != 0]), "N90", width = "250px", digits = 0)
))

## Sample Information

The table below is an aggregation of data for each sample based on their `*.bxstats.gz` file.
Every column after `% valid bx` ignores singletons in its calculations. Hover the column name
to display a tooltip with the column's description.

In [ ]:
df_to_show <- aggregate_df
df_to_show$singletons = round(df_to_show$nonsingletons / (df_to_show$singletons + df_to_show$nonsingletons) * 100, 2)
df_to_show <- df_to_show[, c(-5, -8, -9, -12, -14, -16)]

column_description <- c(
  "name of the sample",
  "total number of alignments",
  "the unique DNA molecules as inferred from linked-read barcodes",
  "molecules composed of two or more single/paired-end sequences, in other words, molecules with linked-read information",
  "number of unique barcodes, which may differ from unique molecules after deconvolution",
  "molecule-to-barcode ratio, which helps benchmark deconvolution performance, if performed",
  "percent of valid barcoded alignments",
  "average number of reads per unique molecule",
  "average percent of a molecule that is covered by a read, where coverage includes unsequenced gaps between linked reads",
  "average percent molecule coverage, where coverage only includes sequences and not the gaps between linked reads",
  "N50 of inferred molecules",
  "N75 of inferred molecules",
  "N90 of inferred molecules"
  )

headerCallback <- c(
  "function(thead, data, start, end, display){",
  paste0(" var tooltips = ['", paste(column_description, collapse = "','"), "'];"),
  "  for(var i=0; i<tooltips.length; i++){",
  "    $('th:eq('+i+')',thead).attr('title', tooltips[i]);",
  "  }",
  "}"
)

DT::datatable(
  df_to_show,
  rownames = F,
  extensions = c('Buttons','FixedColumns'), 
  fillContainer = T,
  colnames = c("sample", "alignments", "unique molecules", "% linked", "unique barcodes", "molecule:bx ratio", "% valid bx", "avg reads/molecule", "avg molecule coverage (linked)", "avg molecule coverage (bp)", "N50", "N75","N90"),
  options = list(
    scrollY = F,
    deferRender = T,
    scrollX = T,
    dom = 'Brtp',
    fixedColumns = list(leftColumns = 1),
    buttons = list(list(extend = "csv",filename = "per_sample_barcode_alignment")),
    headerCallback = JS(headerCallback)
  )
) %>%
  htmlwidgets::onRender(iframe_height(df_to_show))

# it was only for visual purposes, remove it
rm(df_to_show)

## Library Performance
### NX metrics

The **NX**[^1] metrics (e.g. **N50**) tend to be useful for understanding centrality for molecule length in genomic contexts.
With your linked-read sequences aligned to a reference genome, you can make inferences about the lengths of the original molecules from which linked-read fragments were derived. These are the distributions of three common NX metrics (N50, N75, N90) across all samples are given below.

[^1]:
    `NX` is the length of the shortest molecule that when you sum the lengths of every molecule larger than it, represents at least **X%** of the total molecules by length. For example, `N50` would be the molecule length at which the sum of all the molecule lengths larger than it would
    represent **50%** of the total molecules by length (sort of like a median).
    
    As an example, imagine we had molecules with lengths [4, 3, 2, 1], making up a total length of 10. The `N50`
    would be the first number (starting from the biggest) that sums up to at least 5 (50% of 10), which is `3`, because `4` + `3` = 7. The `N90` would be the first number that sums up to at least 9 (90% of 10), which is `2` because `4` + `3` + `2` = 9.

:::{note} Understanding Inferred Molecule Lengths
:class: dropdown
Harpy uses the highest and lowest mapping positions of a read cluster sharing the same barcode (incorporating distance-based deconvolution thresholds, if configured to do so) to infer the original molecule length. Given that it's mostly impossible to understand how much of a source molecule the mapped reads actually covered, it would be more appropriate to think of the inferred molecule lengths (and NX measures) to be more like _"at least this long"_, since the reads likely did not originate from the ends of the source DNA molecule. The absolute length of the source DNA molecule would be a more useful diagnostic in troubleshooting/modifying the actual linked-read chemistry, whereas the the inferred molecule length describes what length of it is actually represented in the sequences and thus more useful in downstream/analytical contexts.
:::

In [ ]:
if(length(unique(aggregate_df$N50)) > 1 && length(unique(aggregate_df$N75)) > 1 && length(unique(aggregate_df$N90)) > 1){
  highchart(height = 400) |>
    hc_add_theme(hc_theme_elementary()) |>
    hc_chart(type = "area", animation = F) |>
    hc_add_series(data = density(aggregate_df$N50), name = "N50", type = "areaspline") |>
    hc_add_series(data = density(aggregate_df$N75), name = "N75", type = "areaspline") |>
    hc_add_series(data = density(aggregate_df$N90), name = "N90", color = "#d3805f", type = "areaspline") |>
    hc_tooltip(enabled = FALSE) |>
    hc_caption(text = "Values derived using non-singleton molecules") |>
    hc_title(text = "NX Stats Across Samples") |>
    hc_xAxis(title = list(text = "length (bp)"), min = 0) |>
    hc_yAxis(title = list(text = "density")) |>
    hc_exporting(enabled = T, filename = "NX.stats",
      buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
    ) |>
    htmlwidgets::onRender(iframe_height(418))
} else if (nrow(aggregate_df) == 1){
  nx_df <- data.frame("NX" = c("N50", "N75", "N90"), "value" = c(aggregate_df$N50, aggregate_df$N75, aggregate_df$N90))
  highchart(width = 400) |>
    hc_chart(type = "column", animation = F) |>
    hc_add_series(nx_df, name = "value", hcaes(x = NX, y = value, color = NX)) |>
    hc_add_theme(hc_theme_elementary()) |>
    hc_tooltip(enabled = TRUE) |>
    hc_caption(text = "Values derived using non-singleton molecules") |>
    hc_title(text = "NX Stats") |>
    hc_xAxis(title = list(text = "Stat"), categories = nx_df$NX) |>
    hc_yAxis(title = list(text = "length (bp)")) |>
    hc_exporting(enabled = T, filename = "NX.stats",
      buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
    ) |>
    htmlwidgets::onRender(iframe_height(418))
} else {
  IRdisplay::display_html(
    sprintf("<p>One of more of the NXX stats don't have enough unique values (minimum: 2) to plot distributions:<br>%s<br>%s<br>%s<p>",
      paste("  N50:", unique(aggregate_df$N50)),
      paste("  N75:", unique(aggregate_df$N75)),
      paste("  N90:", unique(aggregate_df$N90)),
    )
  )
}

### Reads per molecule
This distribution shows the average number of reads per inferred molecule across your samples.

In [ ]:
if(length(unique(aggregate_df$averagereadspermol)) > 1){
  highchart(height = 400) |>
    hc_chart(type = "areaspline", animation = F, groupPadding = 0.0001) |>
    hc_add_theme(hc_theme_elementary()) |>
    hc_add_series(data = density(aggregate_df$averagereadspermol), color = "#e1a42b",
      name = "Reads per Molecule", marker = list(enabled = FALSE)) |>
    hc_title(text = "Average Reads Per Molecule") |>
    hc_xAxis(title = list(text = "reads per molecule (average)")) |>
    hc_yAxis(title = list(text = "density")) |>
    hc_tooltip(enabled = FALSE) |>
    hc_exporting(enabled = T, filename = "reads_per_molecule",
      buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
    ) |>
    htmlwidgets::onRender(iframe_height(418))
} else {
  IRdisplay::display_html("<p>Not enough unique values for reads per molecule to plot distributions (minimum: 2).<p>")
}

### Valid barcodes
This is a distribution of what percent of total alignments each sample had valid barcodes[^2]:

[^2]:
    Valid barcodes with respect to the conventions of a given linked-read chemistry:
    - **haplotagging**: `AxxCxxBxxDxx` where `xx` is not `00`
    - **stLFR**: `X_Y_Z` where `X`, `Y`, or `Z` is not `0`
    - **TELLseq**: `ATCG` where there is no `N` nucleotide

In [ ]:
hs <- hist(
  aggregate_df$totalvalidpercent,
  breaks = seq(0,100, by = 5),
  plot = F
)
hs$counts <- round(hs$counts / sum(hs$counts) * 100, 4)
hs <- data.frame(val = hs$breaks[-1], freq = hs$counts)

highchart(height = 400) |>
  hc_chart(type = "areaspline", animation = F, groupPadding = 0.0001) |>
  hc_add_theme(hc_theme_elementary()) |>
  hc_add_series(data = hs, color = "#8484bd", name = "Percent Alignments", marker = list(enabled = FALSE),
    hcaes(x = val, y = freq)
  ) |>
  hc_title(text = "Percent of Alignments with Valid BX tags") |>
  hc_xAxis(min = 0, max = 100, title = list(text = "% alignments with valid BX tag")) |>
  hc_yAxis(max = 100, title = list(text = "% samples")) |>
  hc_tooltip(crosshairs = TRUE) |>
  hc_exporting(enabled = T, filename = "percent.valid.dist",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  ) |>
  htmlwidgets::onRender(iframe_height(418))

### Linking proportion
This scatterplot is a diagnostic that shows the relationship between the proportion of linked
reads (reads with linked information, aka not singletons) compared to total reads. Each point is a sample and is colored by the
mean number of reads per molecule for that sample.

In [ ]:
highchart(height = 400) |>
  hc_add_theme(hc_theme_elementary()) |>
  hc_chart(type = "scatter", animation = F, groupPadding = 0.0001) |>
  hc_add_series(data = aggregate_df, name = "Linked-Read Performance",
    hcaes(x = totalreads, y = round(nonsingletons / (nonsingletons + singletons), 2), color = averagereadspermol, name = sample, molper = averagereadspermol)
  ) |>
  hc_title(text = "Linked Reads vs Total Reads") |>
  hc_xAxis(title = list(text = "total reads")) |>
  hc_yAxis(max = 1, min = 0, title = list(text = "proportion of reads with linked-read information")) |>
  hc_tooltip(
    crosshairs = FALSE,
    pointFormat= '<b>{point.name}</b><br>reads: <b>{point.x}</b><br>proportion linked reads: <b>{point.y}</b><br>reads per molecule: <b>{point.molper}</b></br>'
    ) |>
  hc_exporting(enabled = T, filename = "linkedread.performance",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  ) |>
  htmlwidgets::onRender(iframe_height(418))

## Per-Sample Metrics
Below is a series of plots showing per-sample metrics. The meaning of percentages and error bars
are provided in the bottom-left captions of the plots.

### Percent valid barcodes

In [ ]:
highchart(height = 400) |>
  hc_chart(type = "xrange", animation = F, groupPadding = 0.0001) |>
  hc_add_theme(hc_theme_elementary()) |>
  hc_add_series(data = aggregate_df, dataLabels = list(enabled = TRUE),
    hcaes(x = 0, x2 = totalreads, y = rev(0:(n_samples-1)), partialFill = totalvalidpercent/100)
  ) |> 
  hc_xAxis(title = list(text = "Total Alignments")) |> 
  hc_yAxis(title = FALSE, gridLineWidth = 0, categories = rev(aggregate_df$sample)) |>
  hc_caption(text = "Percentage represents the percent of alignments with a valid BX barcode, shown in dark green. Light green represents invalid BX barcodes.") |>
  hc_tooltip(enabled = FALSE) |>
  hc_colors(color = "#95d8a7") |>
  hc_title(text = "Percent Valid Barcode") |>
  hc_exporting(enabled = T, filename = "BX.valid.alignments",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  ) |>
    htmlwidgets::onRender(iframe_height(418))

### Percent linked reads

In [ ]:
highchart(height = 400) |>
  hc_add_theme(hc_theme_elementary()) |>
  hc_chart(type = "xrange", animation = F, groupPadding = 0.0001) |>
  hc_add_series(
    data = aggregate_df,
    name = "mean",
    dataLabels = list(enabled = TRUE),
    hcaes(x = 0, x2 = nonsingletons + singletons, y = rev(0:(n_samples-1)), partialFill = round(nonsingletons / (nonsingletons + singletons), 4))
  ) |>
  hc_xAxis(title = list(text = "Total Alignments with Valid Barcodes")) |> 
  hc_yAxis(title = FALSE, gridLineWidth = 0, categories = rev(aggregate_df$sample)) |>
  hc_caption(text = "Percentage represents the percent of molecules composed of more than 2 single-end or paired-end sequences, shown in dark blue. Light blue represents the singletons.") |>
  hc_tooltip(enabled = FALSE) |>
  hc_colors(color = "#8dc6f5") |>
  hc_title(text = "% Linked Sequences") |>
  hc_exporting(enabled = T, filename = "linked",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  ) |>
    htmlwidgets::onRender(iframe_height(418))

### Reads per Molecule

In [ ]:
err_df <- data.frame(x = 0:(n_samples-1), y = aggregate_df$averagereadspermol, low = aggregate_df$averagereadspermol - aggregate_df$sereadspermol, high = aggregate_df$averagereadspermol + aggregate_df$sereadspermol)
highchart(height = 400) |>
  hc_add_theme(hc_theme_elementary()) |>
  hc_chart(inverted=TRUE, animation = F, pointPadding = 0.0001, groupPadding = 0.0001) |>
  hc_add_series(data = aggregate_df, type = "scatter", name = "mean", hcaes(x = 0:(nrow(aggregate_df)-1), y = averagereadspermol), marker = list(radius = 8), color = "#6d73c2", zIndex = 1) |>
  hc_add_series(data = err_df, type = "errorbar", name = "standard error", linkedTo = ":previous", zIndex = 0, stemColor = "#8186c7", whiskerColor = "#8186c7") |>
  hc_xAxis(title = FALSE, gridLineWidth = 0, categories = aggregate_df$sample) |>
  hc_title(text = "Average Reads Per Molecule") |>
  hc_caption(text = "Error bars show the standard error of the mean of non-singleton molecules") |>
  hc_tooltip(crosshairs = TRUE, pointFormat= '<b>{point.y}</b>') |>
  hc_exporting(enabled = T, filename = "avg.reads.per.molecule",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  ) |>
    htmlwidgets::onRender(iframe_height(418))

### Molecule Coverage

In [ ]:
err_df <- data.frame(x = 0:(n_samples-1), y = aggregate_df$averagemolcov, low = aggregate_df$averagemolcov - aggregate_df$semolcov, high = aggregate_df$averagemolcov + aggregate_df$semolcov)
err_df_bp <- data.frame(x = 0:(n_samples-1), y = aggregate_df$averagemolcovbp, low = aggregate_df$averagemolcovbp - aggregate_df$semolcovbp, high = aggregate_df$averagemolcovbp + aggregate_df$semolcovbp)

highchart(height = 400) |>
  hc_add_theme(hc_theme_elementary()) |>
  hc_chart(inverted=TRUE, animation = F, pointPadding = 0.0001, groupPadding = 0.0001) |>
  hc_add_series(data = aggregate_df, type = "scatter", name = "Inferred Fragments", hcaes(x = 0:(nrow(aggregate_df)-1), y = averagemolcov), marker = list(radius = 8), color = "#df77b5", zIndex = 1) |>
  hc_add_series(data = err_df, type = "errorbar", name = "standard error", linkedTo = ":previous", zIndex = 0, stemColor = "#ef94ca", whiskerColor = "#ef94ca") |>
  hc_add_series(data = aggregate_df, type = "scatter", name = "Aligned Base Pairs", hcaes(x = 0:(nrow(aggregate_df)-1), y = averagemolcovbp), marker = list(radius = 8), color = "#924596", zIndex = 1) |>
  hc_add_series(data = err_df_bp, type = "errorbar", name = "standard error", linkedTo = ":previous", zIndex = 0, stemColor = "#916094", whiskerColor = "#916094") |>
  hc_xAxis(title = FALSE, gridLineWidth = 0, categories = aggregate_df$sample) |>
  hc_title(text = "Average % Molecule Coverage") |>
  hc_caption(text = "Error bars show the standard error of the mean percent coverage of non-singleton molecules") |>
  hc_tooltip(crosshairs = TRUE, pointFormat= '<b>{point.y}</b>') |>
  hc_exporting(enabled = T, filename = "avg.molecule.coverage",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  ) |>
    htmlwidgets::onRender(iframe_height(418))